# Chapter 13 - Random Forests

A single decision tree is fast and interpretable but suffers from high variance — small
changes in the training data can produce a completely different tree. **Ensemble methods**
address this by combining many models. The random forest is the most successful
tree-based ensemble: it grows many decorrelated trees on bootstrap samples and averages
their predictions, achieving much lower variance than any individual tree.

**Topics covered:**
- Ensemble methods: why combining models works
- Bagging (Bootstrap Aggregating) concept
- Random Forests: bagging + random feature subsets
- RandomForestClassifier and RandomForestRegressor
- Key hyperparameters: n_estimators, max_features, max_depth, bootstrap
- Out-of-Bag (OOB) score
- Feature importance (mean decrease in impurity, permutation importance)
- Comparing a single tree vs random forest (accuracy and stability)
- Effect of n_estimators on performance
- When random forests fail (extrapolation, high-dimensional sparse)
- Practical example with classification metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.datasets import make_moons, make_classification, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
)
from sklearn.inspection import permutation_importance

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Why Combining Models Works

Consider a toy example: you have 100 independent binary classifiers, each with 60%
accuracy. If you take a majority vote, the probability that the ensemble is correct
follows a binomial distribution. The ensemble accuracy is the probability that more
than half of the classifiers are correct.

By the **law of large numbers**, the ensemble accuracy converges to much higher than any
individual. This works because averaging reduces variance — the individual errors cancel
out, provided the models are **diverse** (i.e., they make different mistakes).

In [ ]:
from scipy.stats import binom

# Ensemble accuracy via majority vote
individual_acc = 0.60
n_classifiers_range = np.arange(1, 501, 2)  # odd numbers to avoid ties

ensemble_accs = []
for n in n_classifiers_range:
    # Probability that more than half are correct
    majority = n // 2 + 1
    p_correct = 1 - binom.cdf(majority - 1, n, individual_acc)
    ensemble_accs.append(p_correct)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_classifiers_range, ensemble_accs, linewidth=2)
ax.axhline(individual_acc, color='red', linestyle='--',
           label=f'Individual accuracy = {individual_acc}')
ax.set_xlabel('Number of classifiers')
ax.set_ylabel('Ensemble accuracy (majority vote)')
ax.set_title('Majority vote ensemble of independent 60%-accurate classifiers')
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

print(f"With 101 classifiers: {ensemble_accs[50]:.4f}")
print(f"With 501 classifiers: {ensemble_accs[-1]:.4f}")

The catch is **independence**: if all classifiers make the same mistakes, averaging does
not help. The art of ensemble methods is creating diverse models — bagging and random
feature selection achieve this for trees.

---
## 2. Bagging (Bootstrap Aggregating)

**Bagging** was introduced by Leo Breiman in 1996. The idea:

1. Draw $B$ **bootstrap samples** (random samples with replacement) from the training set.
2. Train a separate model (typically a decision tree) on each bootstrap sample.
3. For prediction, average the outputs (regression) or take a majority vote (classification).

Each bootstrap sample contains about 63.2% of the unique training instances (the rest
are duplicates). The ~36.8% of instances **not** selected form the **out-of-bag (OOB)**
set for that tree, which serves as a free validation set.

In [ ]:
# Demonstrate bootstrap sampling
n_samples = 1000
n_bootstrap = 10000

unique_fractions = []
for _ in range(n_bootstrap):
    bootstrap_idx = np.random.choice(n_samples, size=n_samples, replace=True)
    unique_fractions.append(len(np.unique(bootstrap_idx)) / n_samples)

print(f"Average fraction of unique samples in a bootstrap: {np.mean(unique_fractions):.4f}")
print(f"Theoretical value: 1 - (1 - 1/n)^n -> 1 - 1/e = {1 - np.exp(-1):.4f}")

In [ ]:
# Bagging with sklearn's BaggingClassifier
X_moons, y_moons = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=42
)

# Single tree vs bagged trees
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)

print(f"Single tree — Train: {single_tree.score(X_train, y_train):.4f}  "
      f"Test: {single_tree.score(X_test, y_test):.4f}")
print(f"Bagging (100 trees) — Train: {bagging.score(X_train, y_train):.4f}  "
      f"Test: {bagging.score(X_test, y_test):.4f}")

In [ ]:
# Visualise single tree vs bagging decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

for ax, model, name in zip(axes,
                           [single_tree, bagging],
                           ['Single tree', 'Bagging (100 trees)']):
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'{name}\nTest accuracy: {model.score(X_test, y_test):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

The bagged ensemble produces smoother decision boundaries because the individual trees'
idiosyncratic decisions average out.

---
## 3. From Bagging to Random Forests

Bagging reduces variance, but there is a problem: if one feature is very strong, every
tree will split on that feature first, making the trees **correlated**. Correlated trees
reduce the benefit of averaging.

**Random forests** (Breiman, 2001) fix this by adding a second layer of randomness:
at each split, only a **random subset of features** is considered. This forces each tree
to explore different parts of feature space, producing more diverse trees.

The default `max_features` is:
- **Classification:** `sqrt(n_features)`
- **Regression:** `n_features` (all features, but still with bootstrap randomness)

In [ ]:
# Random Forest on make_moons
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print(f"Random Forest — Train: {rf.score(X_train, y_train):.4f}  "
      f"Test: {rf.score(X_test, y_test):.4f}")

In [ ]:
# Compare: single tree, bagging, random forest
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ('Single tree', single_tree),
    ('Bagging (100 trees)', bagging),
    ('Random Forest (100 trees)', rf),
]

for ax, (name, model) in zip(axes, models):
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'{name}\nTest: {model.score(X_test, y_test):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Single tree vs bagging vs random forest', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Key Hyperparameters

| Parameter | Description | Typical values |
|-----------|-------------|----------------|
| `n_estimators` | Number of trees in the forest | 100 - 1000+ |
| `max_features` | Features considered at each split | `'sqrt'` (classification), `1.0` (regression) |
| `max_depth` | Maximum depth of each tree | None (unlimited) or 10 - 30 |
| `min_samples_split` | Minimum samples to split a node | 2 - 20 |
| `min_samples_leaf` | Minimum samples in each leaf | 1 - 10 |
| `bootstrap` | Whether to use bootstrap samples | True (default) |
| `n_jobs` | Number of parallel jobs | -1 (use all cores) |

More trees almost always improve performance (up to a point), and they never cause
overfitting — only increasing computational cost.

---
## 5. Out-of-Bag (OOB) Score

Each tree sees only ~63% of the training data. The remaining ~37% can be used as a
validation set **for that tree**. By aggregating predictions from each tree only on its
OOB samples, we get an estimate of generalisation error without needing a separate
validation set or cross-validation.

Set `oob_score=True` to compute it.

In [ ]:
rf_oob = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_oob.fit(X_train, y_train)

print(f"OOB score:      {rf_oob.oob_score_:.4f}")
print(f"Test accuracy:  {rf_oob.score(X_test, y_test):.4f}")
print(f"\nThe OOB score is a good approximation of the test score,")
print(f"without requiring a separate holdout set.")

---
## 6. Effect of n_estimators on Performance

Adding more trees reduces variance. After a certain point, the benefit saturates and
performance plateaus. Unlike boosting, more trees in a random forest **never cause
overfitting** — they just cost more compute.

In [ ]:
n_estimators_range = [1, 2, 5, 10, 20, 50, 100, 200, 300, 500]
oob_scores = []
test_scores = []

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    test_scores.append(rf_temp.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_estimators_range, oob_scores, 'o-', label='OOB score', linewidth=2)
ax.plot(n_estimators_range, test_scores, 's-', label='Test accuracy', linewidth=2)
ax.set_xlabel('n_estimators')
ax.set_ylabel('Accuracy')
ax.set_title('Random forest performance vs number of trees')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()

Performance improves rapidly with the first few dozen trees, then plateaus. In practice,
100-500 trees is usually sufficient.

---
## 7. Feature Importance: Impurity-Based

Random forests aggregate feature importance across all trees. The **mean decrease in
impurity (MDI)** measures how much each feature contributes to reducing Gini impurity
(or MSE for regression) across all splits, averaged over all trees.

This is fast to compute but has a known bias: it favours features with **many unique
values** (high cardinality) and features that are used in many splits simply because
they happen to appear in many trees.

In [ ]:
# Feature importance on the Wine dataset
wine = load_wine()
X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42
)

rf_wine = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_wine.fit(X_w_train, y_w_train)

print(f"Test accuracy: {rf_wine.score(X_w_test, y_w_test):.4f}")

# Plot impurity-based importance
importances = rf_wine.feature_importances_
idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(np.array(wine.feature_names)[idx], importances[idx], color='steelblue')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Impurity-based feature importance (Random Forest, Wine dataset)')
plt.tight_layout()
plt.show()

---
## 8. Feature Importance: Permutation-Based

**Permutation importance** is a more reliable measure. The idea:

1. Train the model and record its baseline score on the test set.
2. For each feature, randomly shuffle (permute) its values and measure how much the
   score drops.
3. A large drop means the feature is important; a small drop means the model does not
   rely on it.

Unlike MDI, permutation importance is computed on held-out data and does not suffer
from the cardinality bias.

In [ ]:
# Permutation importance
perm_result = permutation_importance(
    rf_wine, X_w_test, y_w_test, n_repeats=30, random_state=42, n_jobs=-1
)

perm_idx = perm_result.importances_mean.argsort()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Impurity-based
axes[0].barh(np.array(wine.feature_names)[idx], importances[idx], color='steelblue')
axes[0].set_xlabel('Mean Decrease in Impurity')
axes[0].set_title('Impurity-based importance')

# Permutation-based
axes[1].barh(
    np.array(wine.feature_names)[perm_idx],
    perm_result.importances_mean[perm_idx],
    xerr=perm_result.importances_std[perm_idx],
    color='coral'
)
axes[1].set_xlabel('Mean accuracy decrease')
axes[1].set_title('Permutation importance (on test set)')

plt.suptitle('Impurity-based vs permutation feature importance', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The rankings may differ, especially for correlated features. Permutation importance is
generally preferred for model interpretation because it is measured on held-out data
and accounts for the model's actual reliance on each feature.

---
## 9. Single Tree vs Random Forest: Accuracy and Stability

One of the key advantages of random forests over single trees is **stability**. Let us
train both on multiple random train/test splits and compare the distribution of
test scores.

In [ ]:
# Stability comparison
n_trials = 50
tree_scores = []
rf_scores = []

X_full, y_full = make_moons(n_samples=500, noise=0.25, random_state=42)

for i in range(n_trials):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_full, y_full, test_size=0.25, random_state=i
    )

    tree = DecisionTreeClassifier(random_state=42)
    tree.fit(X_tr, y_tr)
    tree_scores.append(tree.score(X_te, y_te))

    forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    forest.fit(X_tr, y_tr)
    rf_scores.append(forest.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(8, 5))
positions = [1, 2]
bp = ax.boxplot([tree_scores, rf_scores], positions=positions, widths=0.5,
                patch_artist=True)
bp['boxes'][0].set_facecolor('lightcoral')
bp['boxes'][1].set_facecolor('steelblue')
ax.set_xticks(positions)
ax.set_xticklabels(['Single Tree', 'Random Forest (100)'])
ax.set_ylabel('Test Accuracy')
ax.set_title(f'Stability comparison over {n_trials} random splits')

# Annotate with mean and std
for pos, scores, color in zip(positions, [tree_scores, rf_scores],
                               ['red', 'blue']):
    ax.text(pos, np.max(scores) + 0.01,
            f'mean={np.mean(scores):.3f}\nstd={np.std(scores):.3f}',
            ha='center', fontsize=10, color=color)

plt.tight_layout()
plt.show()

The random forest achieves higher accuracy on average and has much lower variance across
different random splits. This stability is one of its greatest practical advantages.

---
## 10. Random Forest Regression

Random forests work equally well for regression. Each tree predicts the mean of its
leaf, and the forest averages all tree predictions.

In [ ]:
# Regression example
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(300, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + 0.3 * np.random.randn(300)

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

# Single tree vs random forest
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_tr_r, y_tr_r)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_tr_r, y_tr_r)

X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name in zip(axes,
                           [tree_reg, rf_reg],
                           ['Single Tree', 'Random Forest (100 trees)']):
    y_pred = model.predict(X_plot)
    ax.scatter(X_tr_r, y_tr_r, s=15, alpha=0.5, label='Train')
    ax.plot(X_plot, y_pred, color='red', linewidth=2, label='Prediction')
    ax.plot(X_plot, np.sin(X_plot), color='green', linewidth=1.5,
            linestyle='--', label='True function')
    mse = mean_squared_error(y_te_r, model.predict(X_te_r))
    ax.set_title(f'{name}\nTest MSE: {mse:.4f}')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()

The forest prediction is much smoother than the single tree because it averages many
step functions. However, the result is still a step function — forests cannot produce
truly smooth curves.

---
## 11. When Random Forests Fail

Random forests are remarkably robust, but they have notable limitations:

### Extrapolation
Trees (and therefore forests) cannot extrapolate beyond the range of the training data.
Each prediction is a weighted average of training targets, so the output is always
bounded by the training range.

### High-dimensional sparse data
When most features are irrelevant (e.g., text data with thousands of sparse features),
the random feature subsetting can cause trees to repeatedly pick uninformative features,
degrading performance.

### Linear relationships
A simple linear relationship requires many axis-aligned splits to approximate. A linear
model would do the job with a single coefficient.

In [ ]:
# Demonstrate extrapolation failure
np.random.seed(42)
X_extrap = np.linspace(0, 8, 200).reshape(-1, 1)
y_extrap = 0.5 * X_extrap.ravel() + np.sin(X_extrap.ravel()) + 0.2 * np.random.randn(200)

# Train on first 150 points (x in [0, 6]), predict on all
X_tr_ex = X_extrap[:150]
y_tr_ex = y_extrap[:150]

rf_extrap = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_extrap.fit(X_tr_ex, y_tr_ex)

X_full_plot = np.linspace(0, 10, 500).reshape(-1, 1)
y_pred_ex = rf_extrap.predict(X_full_plot)
y_true = 0.5 * X_full_plot.ravel() + np.sin(X_full_plot.ravel())

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_tr_ex, y_tr_ex, s=15, alpha=0.5, label='Training data')
ax.plot(X_full_plot, y_pred_ex, color='red', linewidth=2, label='RF prediction')
ax.plot(X_full_plot, y_true, color='green', linewidth=1.5, linestyle='--',
        label='True function')
ax.axvline(X_tr_ex[-1], color='grey', linestyle=':', label='Training boundary')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Random forest cannot extrapolate beyond training data range')
ax.legend()
plt.tight_layout()
plt.show()

Beyond the training data boundary, the forest prediction **flattens out** — it simply
repeats the average of the nearest leaf values. This is a fundamental limitation of
all tree-based methods.

---
## 12. Practical Example: Classification with Full Metrics

Let us put it all together with a higher-dimensional dataset and a proper evaluation.

In [ ]:
# Generate a challenging classification dataset
X_clf, y_clf = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=3, n_clusters_per_class=2,
    random_state=42
)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42
)

# Single tree
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_tr_c, y_tr_c)

# Random forest
rf_clf = RandomForestClassifier(
    n_estimators=200, oob_score=True, random_state=42, n_jobs=-1
)
rf_clf.fit(X_tr_c, y_tr_c)

print("=== Single Decision Tree ===")
print(f"Train accuracy: {tree_clf.score(X_tr_c, y_tr_c):.4f}")
print(f"Test accuracy:  {tree_clf.score(X_te_c, y_te_c):.4f}")
print()
print("=== Random Forest (200 trees) ===")
print(f"Train accuracy: {rf_clf.score(X_tr_c, y_tr_c):.4f}")
print(f"OOB score:      {rf_clf.oob_score_:.4f}")
print(f"Test accuracy:  {rf_clf.score(X_te_c, y_te_c):.4f}")

In [ ]:
# Detailed classification report for the random forest
y_pred_rf = rf_clf.predict(X_te_c)
print(classification_report(y_te_c, y_pred_rf, digits=3))

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in zip(axes,
                           [tree_clf, rf_clf],
                           ['Single Tree', 'Random Forest']):
    cm = confusion_matrix(y_te_c, model.predict(X_te_c))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Class 0', 'Class 1', 'Class 2'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAccuracy: {model.score(X_te_c, y_te_c):.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for the 20-dimensional dataset
perm_result_clf = permutation_importance(
    rf_clf, X_te_c, y_te_c, n_repeats=20, random_state=42, n_jobs=-1
)

sorted_idx = perm_result_clf.importances_mean.argsort()

fig, ax = plt.subplots(figsize=(8, 8))
ax.boxplot(
    perm_result_clf.importances[sorted_idx].T,
    vert=False,
    labels=[f'Feature {i}' for i in sorted_idx]
)
ax.set_xlabel('Decrease in accuracy')
ax.set_title('Permutation importance (Random Forest, 20-feature dataset)')
plt.tight_layout()
plt.show()

---
## 13. Summary

| Concept | Key takeaway |
|---------|-------------|
| **Bagging** | Reduces variance by training trees on bootstrap samples and averaging |
| **Random Forest** | Bagging + random feature subsets at each split for decorrelated trees |
| **n_estimators** | More trees is better (up to diminishing returns); never causes overfitting |
| **OOB score** | Free validation estimate from the ~37% of samples each tree does not see |
| **Feature importance** | MDI is fast but biased; permutation importance is more reliable |
| **Strengths** | Robust, parallel, few hyperparameters to tune, works well out of the box |
| **Weaknesses** | Cannot extrapolate, slow for very large n_estimators, not as interpretable as a single tree |

Random forests are often the first model to try on a new tabular dataset. They provide
a strong baseline with minimal tuning. For even higher accuracy, gradient boosting
(next notebook) often outperforms random forests by building trees sequentially rather
than independently.

---

**Next up:** [03 - Boosting](03_boosting.ipynb) — building trees sequentially so each
one learns from the mistakes of its predecessors.